## Load Historical Training and Later Evaluation Data

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import duckdb

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    average_precision_score,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

con = duckdb.connect()

training_data = con.read_parquet(
    str(
        PROCESSED_DIR
        / "historical_training_features_cutoff_20161130.parquet"
    )
).df()

evaluation_data = con.read_parquet(
    str(
        PROCESSED_DIR
        / "candidate_transaction_features_v2_cutoff_20170228.parquet"
    )
).df()

feature_columns = [
    "transaction_rows_90d",
    "transaction_days_90d",
    "cancel_rows_90d",
    "auto_renew_share_90d",
    "avg_plan_days_90d",
    "days_since_last_transaction_90d",
    "has_transaction_90d",
]

X_train = training_data[feature_columns].astype("float64")
y_train = training_data["is_churn"].astype("int8")

X_eval = evaluation_data[feature_columns].astype("float64")
y_eval = evaluation_data["is_churn"].astype("int8")

assert training_data["msno"].is_unique
assert evaluation_data["msno"].is_unique

print("Training rows:", len(X_train))
print("Evaluation rows:", len(X_eval))

Training rows: 882127
Evaluation rows: 970960


## Define Untuned Baseline Models

In [2]:
models = {
    "Logistic regression": Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=True),
        ),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(max_iter=1000, random_state=42),
        ),
    ]),
    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=100,
        tree_method="hist",
        random_state=42,
        n_jobs=4,
    ),
    "LightGBM": LGBMClassifier(
        objective="binary",
        n_estimators=100,
        random_state=42,
        n_jobs=4,
    ),
}

## Fit Models and Compare Later-Cohort Performance

In [3]:
comparison_rows = []

for name, model in models.items():
    print(f"Training {name}...", flush=True)

    start = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - start

    probabilities = model.predict_proba(X_eval)[:, 1]

    assert np.isfinite(probabilities).all()
    assert (
        (probabilities >= 0)
        & (probabilities <= 1)
    ).all()

    comparison_rows.append({
        "model": name,
        "log_loss": log_loss(y_eval, probabilities),
        "roc_auc": roc_auc_score(y_eval, probabilities),
        "average_precision": average_precision_score(
            y_eval, probabilities
        ),
        "fit_seconds": fit_seconds,
    })

    print(f"Finished {name} in {fit_seconds:.1f} seconds.")

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("log_loss")
    .reset_index(drop=True)
)

model_comparison

Training Logistic regression...
Finished Logistic regression in 0.8 seconds.
Training XGBoost...
Finished XGBoost in 1.2 seconds.
Training LightGBM...
[LightGBM] [Info] Number of positive: 85237, number of negative: 796890
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014798 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 159
[LightGBM] [Info] Number of data points in the train set: 882127, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.096627 -> initscore=-2.235281
[LightGBM] [Info] Start training from score -2.235281
Finished LightGBM in 1.3 seconds.


,model,log_loss,roc_auc,average_precision,fit_seconds
0,Logistic regression,0.212926,0.829580,0.536523,0.807287
1,XGBoost,0.229378,0.828912,0.490584,1.233205
2,LightGBM,0.229887,0.829479,0.488706,1.269828


In [4]:
import json
import platform
import sklearn
import xgboost
import lightgbm

REPORT_DIR.mkdir(parents=True, exist_ok=True)

model_comparison.to_csv(
    REPORT_DIR / "untuned_model_comparison.csv",
    index=False,
)

model_parameters = {
    name: model.get_params(deep=True)
    for name, model in models.items()
}

benchmark_config = {
    "evaluation_method": "historical cohort evaluation",
    "training_feature_cutoff": "2016-11-30",
    "training_outcome_end": "2017-01-30",
    "evaluation_feature_cutoff": "2017-02-28",
    "training_rows": len(X_train),
    "evaluation_rows": len(X_eval),
    "features": feature_columns,
    "hyperparameter_tuning_completed": False,
    "early_stopping_used": False,
    "independent_test_evaluation_completed": False,
    "versions": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "lightgbm": lightgbm.__version__,
    },
    "model_parameters": model_parameters,
}

(REPORT_DIR / "untuned_benchmark_config.json").write_text(
    json.dumps(benchmark_config, indent=2, default=str) + "\n",
    encoding="utf-8",
)

print("Untuned benchmark results and configuration saved.")

Untuned benchmark results and configuration saved.
